In [1]:
import os
import datetime
import warnings
import polars as pl
import pandas as pd
import altair as alt

from src.najdi_rok import najdi_rok
from src.pocet_stran import pocet_stran
from src.bez_bordelu import bez_bordelu
from src.alt_friendly import alt_friendly
from src.hezke_jmeno import hezke_jmeno
from src.kristi_promin import kristi_promin
from src.zjisti_vazbu import zjisti_vazbu
from src.me_to_neurazi import me_to_neurazi

pl.Config(tbl_rows=100)
alt.data_transformers.disable_max_rows()
alt.themes.register('irozhlas', kristi_promin)
alt.themes.enable('irozhlas')
warnings.filterwarnings('ignore')

In [2]:
df = pl.read_parquet(os.path.join("data/cnb_sloupce","leader.parquet"))
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","100.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","008.parquet")), left_on="001", right_on="001", how="left")
df = df.to_pandas()
df = df[df["leader"].str[6].isin(["a", "t"])]
df = df[~df["leader"].str[7].isin(["b", "i", "s", " "])]
df = df[(df["008"].str[15:17] == "xr") & (df["008"].str[35:38] == "cze")]
df = pl.from_pandas(df)
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","020.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","022.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","245.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","300.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","655.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","700.parquet")), left_on="001", right_on="001", how="left")
df = df.explode("022_a").filter(pl.col("022_a").is_null())
df = df.with_columns(pl.col('008').map_elements(najdi_rok, return_dtype=int).alias('rok'))
df = df.with_columns(pl.col('300_a').map_elements(pocet_stran, return_dtype=int).alias('stran'))
df = df.with_columns(pl.col('245_a').map_elements(bez_bordelu, return_dtype=str))
df = df.explode("020_q").with_columns(pl.col("020_q").map_elements(zjisti_vazbu, return_dtype=str).alias('vazba'))
df = df.explode('245_p').with_columns(pl.col('245_p').map_elements(bez_bordelu, return_dtype=str))
print(len(df))
df = df.filter(~pl.col('rok').is_null()).sort(by='rok')
df = df.filter((~pl.col("245_h").str.contains("grafika")) | pl.col("245_h").is_null())
print(len(df))

1001279
991612


In [3]:
df = df.filter(pl.col("rok") >= 1800)

In [4]:
df.group_by(["100_a","100_7"]).len().sort(by="len",descending=True).head(n=100)

100_a,100_7,len
str,str,u32
null,null,202742
"""Unger, Gert F.,""","""jn20001103529""",1682
"""Vandenberg, Patricia,""","""jn20000810141""",1185
"""Němcová, Božena,""","""jk01083016""",1126
"""Jirásek, Alois,""","""jk01051816""",1112
"""Čapek, Karel,""","""jk01021023""",874
"""Shakespeare, William,""","""jn19981002129""",822
"""Verne, Jules,""","""jn19990008769""",793
"""Dark, Jason,""","""jn20000810032""",725


In [5]:
nechcemejetam = [
    "jn19990008769",
    "jx20060515016",
    "jn19981002230",
    "jn19981001737",
    "jn19990210182",
    "jn19990001842",
    "jn19990002786",
    "jn19990004346",
    "jn20020721077",
    "jn19990210513",
    "jn19990005488",
    "jo20000080627",
    "jn19990000171",
    "jn20001005715",
    "jn19981002409",
    "jn20000810141",
    "jn19981002129",
    "jn20001103529",
    "jn20000810032",
    "jn19990001513",
    "jx20040611003",
    "jn19990005499",
    "jn19981002230",
    "jn19990001907",
    "jo2005267810",
    "jo20241218643"
]

In [6]:
df = df.filter(~pl.col("100_7").is_in(nechcemejetam))

In [7]:
df.group_by("rok").len().sort(by="rok")

rok,len
i64,u32
1800,1
1801,10
1802,10
1803,14
1804,15
1805,16
1806,9
1807,16
1808,16


In [8]:
df.group_by("100_a").len().sort(by="len",descending=True).head(100)

100_a,len
str,u32
"""Němcová, Božena,""",1126
"""Jirásek, Alois,""",1112
"""Čapek, Karel,""",880
"""Vedral, Jiří,""",714
"""Javořická, Vlasta,""",685
"""Neruda, Jan,""",666
"""Erben, Karel Jaromír,""",543
"""Vrchlický, Jaroslav,""",533
"""Masaryk, Tomáš Garrigue,""",499


In [9]:
df.filter(pl.col("100_a") == "Mark, William,")

leader,001,100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,008,020_q,020_c,020_a,020_z,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,vazba
str,str,str,str,str,list[str],str,str,list[str],str,str,str,str,list[str],list[str],list[str],str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str


In [10]:
len(df)

728131

In [11]:
df = df.explode("100_4").filter(pl.col("100_4") == "aut")

In [12]:
len(df)

691262

In [13]:
df = df.unique(subset=["100_a","245_a"])

In [14]:
len(df)

485690

In [15]:
len(list(set(df.select(pl.col("100_7")).to_series().to_list())))

147909

In [16]:
df = df.with_columns(pl.col("100_a").map_elements(hezke_jmeno).alias("jméno"))

In [17]:
df.group_by("jméno").len().sort(by="len",descending=True).head(11)

jméno,len
str,u32
"""Jiří Vedral""",656
"""Božena Němcová""",274
"""Alexandr Batěk""",261
"""Josef Šváb-Malostranský""",260
"""Jan Amos Komenský""",258
"""Zuzana Pospíšilová""",249
"""Jan Neruda""",236
"""Jaroslav Vrchlický""",233
"""Tomáš Garrigue Masaryk""",209


In [18]:
hvezdy = df.group_by("100_7").len().sort(by="len",descending=True).head(10).select(pl.col("100_7")).to_series().to_list()
hvezdy

['mzk2003169026',
 'jk01083016',
 'jk01011106',
 'jk01131780',
 'jk01061444',
 'mzk2006331486',
 'jk01083209',
 'jk01151037',
 'jk01080472',
 'jk01131099']

In [19]:
import datetime

In [20]:
aut = pl.read_parquet(os.path.join("data","aut_vyber.parquet"))

In [21]:
mrtvi = aut.explode("100_7").filter(pl.col("100_7").is_in(hvezdy)).explode("046_g").with_columns(pl.col("046_g").map_elements(lambda x: int(x)).alias("umrti")).select(pl.col(["100_7","umrti"])).filter(pl.col('umrti').is_between(1800,2025)).with_columns(pl.col("umrti").map_elements(lambda x: datetime.date(year=int(x), month=1, day=1), return_dtype=pl.Date).cast(pl.Datetime))

In [22]:
do_grafu = df.filter(
    pl.col("100_7").is_in(hvezdy)
).with_columns(
    pl.col("rok").map_elements(lambda x: datetime.date(year=int(x), month=1, day=1), return_dtype=pl.Date).cast(pl.Datetime)
).join(
    df.group_by('100_7').len(), on='100_7', how='left'
).filter(
    pl.col("len") > 10
).with_columns(
    pl.col("len").map_elements(lambda x: str(x) + "×")
).join(
    mrtvi, how='left', on='100_7'
).with_columns(
    pl.concat_str([pl.col('len'), pl.col('jméno')], separator=' ').alias('jméno')
)

In [23]:
do_grafu

leader,001,100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,008,020_q,020_c,020_a,020_z,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,…,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,vazba,jméno,len,umrti
str,str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,…,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],datetime[μs],i64,str,str,str,datetime[μs]
""" nam a22 a 4500""","""nkc20081808526""","""1""","""Komenský, Jan Amos,""","""jk01061444""","""aut""","""1592-1670""",null,null,null,null,"""080613s1874 xr e 0…","""(Brož.)""",null,null,null,null,null,null,null,null,"""1""","""0""","""J.A. Komenského Řeč o vzděláva…",null,"""z latiny vyložil Fr.J. Zoubek""",null,null,null,null,null,"[""22 s. ;""]",null,"[""27 cm""]",null,null,null,…,"[""fd133137""]","[""czenas""]",null,null,null,"[""1""]","[""Zoubek, František Jan,""]","[""trl""]","[""1832-1890""]","[""jk01152785""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1874-01-01 00:00:00,22,"""brožovaná""","""258× Jan Amos Komenský""","""258×""",null
""" nam a22 1 4500""","""bknhak09095""","""1""","""Němcová, Božena,""","""jk01083016""","""aut""","""1820-1862""",null,null,null,null,"""030517s1933 xr |…",null,null,null,null,null,null,null,null,null,"""1""","""0""","""Světská krása""","""Slovenská národní pohádka /""","""Božena Němcová""",null,null,null,null,null,"[""19 s. ;""]",null,"[""4°""]",null,null,null,…,"[""fd133054""]","[""czenas""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1933-01-01 00:00:00,19,null,"""274× Božena Němcová""","""274×""",1862-01-01 00:00:00
""" nam a22 a 4500""","""cpk20031191582""","""1""","""Vedral, Jiří,""","""mzk2003169026""","""aut""","""1973-""",null,null,null,null,"""030717s2002 xr e d 0…","""(brož.) :""","[""Kč 30,00""]","[""80-86261-79-4""]",null,null,null,null,null,null,"""1""","""0""","""Anglicko-český slovník - proje…",null,"""J. Vedral""",null,null,null,null,null,"[""16 s. ;""]",null,"[""21 cm""]",null,null,null,…,"[""fd132493"", null]","[""czenas"", ""eczenas""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2002-01-01 00:00:00,16,"""brožovaná""","""655× Jiří Vedral""","""655×""",null
""" nam a22 1 4500""","""nos190249457""","""1""","""Šváb-Malostranský, Josef,""","""jk01131780""","""aut""","""1860-1932""",null,null,null,null,"""001127s1902 xr …",null,null,null,null,null,null,null,null,null,"""1""","""0""","""Ubrečená""","""deklamace pro dámu anebo pána …","""napsal a panu Al. Chamradovi-S…",null,null,null,null,null,"[""6 s. ;""]",null,"[""23 cm""]",null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1902-01-01 00:00:00,6,null,"""260× Josef Šváb-Malostranský""","""260×""",1932-01-01 00:00:00
""" cam a22 a 4500""","""nkc20091997238""","""1""","""Vedral, Jiří,""","""mzk2003169026""","""aut""","""1973-""",null,null,null,null,"""090803s2009 xr e d 0…","""(brož.)""",null,"[""978-80-87345-05-4""]",null,null,null,null,null,null,"""1""","""0""","""Biologie pro překladatele 2. -…",null,"""J. Vedral""",null,null,null,nul

In [24]:
hvezdy_poradi = do_grafu.group_by('jméno').len().sort(by="len",descending=True).to_series().to_list()
hvezdy_poradi

['655× Jiří Vedral',
 '274× Božena Němcová',
 '261× Alexandr Batěk',
 '260× Josef Šváb-Malostranský',
 '258× Jan Amos Komenský',
 '249× Zuzana Pospíšilová',
 '236× Jan Neruda',
 '233× Jaroslav Vrchlický',
 '209× Tomáš Garrigue Masaryk',
 '205× Alfons Bohumil Šťastný']

In [25]:
y_encoding = {
        'field': 'jméno',
        'type': 'nominal',
        'title': None,
        'sort': hvezdy
    }

In [26]:
len(do_grafu)

2840

In [27]:
do_grafu.filter(pl.col("jméno") == "Božena Němcová").group_by("245_a").len()

245_a,len
str,u32


In [28]:
import json
from src.me_to_neurazi import me_to_neurazi
with open(os.path.join('src','kredity.json'), 'r', encoding='utf-8') as kredity:
    kredity = json.loads(kredity.read())

In [29]:
base = alt.Chart(do_grafu.to_pandas(), 
                     title=alt.Title(f"{len(hvezdy)} nejvydávanějších autorů a autorek (bez reprintů)",
                                     subtitle=["Co tečka, to kniha. Černá čárka označuje rok úmrtí."]))

kulicky = base.mark_circle(size=6, opacity=1).encode( 
            x=alt.X("rok:T", title=None, axis=alt.Axis(domainOpacity=0, tickColor='#DCDDD6', tickCount=5)), 
            y=alt.Y("jméno:N", sort=hvezdy, title=None, axis=alt.Axis(orient='left', domainOpacity=0, tickColor='white', labelExpr='split(datum.label, "× ")[1]')), 
            yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(range=[3, 15])), 
            color=alt.Color('jméno:N', scale=alt.Scale(range=['#E09DA3']), 
                            sort=hvezdy_poradi).legend(None)) \
        .transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")

kdy_umreli = base.mark_tick(
    color='#292829',  # optional: you can specify color
    thickness=1.5,
    height=9
).encode(
    x=alt.X('umrti:T', title=None),
    y=alt.Y("jméno:N", sort=hvezdy_poradi, title=None, axis=alt.Axis(orient='left', tickColor='white', labels=False)))

pocty = base.mark_circle(size=0).encode(x=alt.X('rok:T', title=None), 
    y=alt.Y('jméno:N', title=None, sort=hvezdy_poradi, axis=alt.Axis(orient="right", tickColor='white', labelExpr='split(datum.label, " ")[0]')))

zebricek = alt.layer(kulicky, pocty, kdy_umreli).properties(
    width=kredity['sirka'] * 1.2,
    autosize={'type': 'fit', 'contains': 'padding'}
).configure_view(stroke='transparent')

zebricek

alt.LayerChart(...)

In [30]:
me_to_neurazi(zebricek, kredity=kredity['default'], soubor="02_nejvydavanejsi_autorstvo")

<figure>
    <a href="https://data.irozhlas.cz/knihy-grafy/02_nejvydavanejsi_autorstvo.svg" target="_blank">
    <img src="https://data.irozhlas.cz/knihy-grafy/02_nejvydavanejsi_autorstvo.svg" width="100%" alt="Omlouváme se, ale alternativní text se nepodařilo vygenerovat. Texty v grafu by měly být čitelné ze zdrojového souboru SVG." />
    </a>
    </figure>


In [31]:
df.filter(pl.col("100_a") == "Pospíšilová, Zuzana,").group_by('rok').len().sort(by='rok')

rok,len
i64,u32
2005,2
2006,7
2007,7
2008,12
2009,12
2010,19
2011,20
2012,24
2013,11


In [32]:
df.filter(pl.col("100_a") == "Pospíšilová, Zuzana,").group_by('rok').len().sort(by='rok').filter(pl.col("rok").is_between(2019,2024)).select(pl.col("len")).median()

len
f64
9.0


In [33]:
(308-253)/9

6.111111111111111

In [34]:
df.filter(pl.col("100_a") == "Pospíšilová, Zuzana,").select(pl.col(['245_a','rok']))

245_a,rok
str,i64
"""Pohádkové uspávanky""",2016
"""Ve škole""",2007
"""Lesní pohádky""",2014
"""Fánkova další dobrodružství""",2011
"""Hravá autoškola""",2012
"""Školnice Valerie v podezření""",2017
"""O kachničce od rybníčka""",2010
"""Barevný svět""",2014
"""Co už víme o zvířátkách""",2008


In [35]:
vedral = df.filter(pl.col("100_a") == "Vedral, Jiří,").select(pl.col("245_a")).to_series().to_list()
vedral

['Anglicko-český slovník - projektování, organizace a ekonomika',
 'Biologie pro překladatele 2. - němčina',
 'Slovensko-český slovník jmen ptáků',
 'Čínsko-český slovník jmen rostlin',
 'Bosensko-český slovník veřejných zakázek CPV',
 'Chemie pro překladatele 2.',
 'Seversko-český přírodovědný slovník',
 'Česko-italský slovník jmen ptáků',
 'Anglicko-český slovník - ministři a ústavní činitelé',
 'Nizozemsko-český slovník veřejných zakázek CPV',
 'Maltsko-český slovník veřejných zakázek CPV',
 'Lotyšsko-český celní sazebník',
 'Anglicko-český polygrafický slovník',
 'Chorvatsko-český slovník PRODCOM',
 'Norsko-český slovník zaměstnanosti ISCO 08',
 'Česko-německý slovník kamenoprůmyslu',
 'Španělsko-český slovník ekonomických činností (NACE)',
 'Lotyšsko-český slovník ekonomických činností (NACE)',
 'Japonsko-český zemědělský slovník',
 'Dánsko-český biologický slovník',
 'Makedonsko-český celní sazebník',
 'Německo-český slovník PRODCOM na CD',
 'Slovensko-český slovník zaměstnání IS

In [36]:
[x for x in vedral if "chemie" in x.lower()]

['Chemie pro překladatele 2.',
 'Chemie pro překladatele - italština',
 'Chemie III.',
 'Chemie V.',
 'Chemie pro překladatele - španělština',
 'Chemie pro překladatele - jazyky bývalé Jugoslávie',
 'Chemie I.',
 'Chemie VI.',
 'Chemie II.',
 'Chemie IV.',
 'Chemie pro překladatele']

In [37]:
[x for x in vedral if "sex" in x.lower()]

['Anglicko-český slovník sexu', 'Anglicko-český slovník sexu na CD']

In [38]:
with open(os.path.join("data_raw","vedral.json"), "w+", encoding="utf-8") as v:
    v.write(json.dumps(vedral))